# AutoMixAI – ANN Beat Detection Model Training

Train a Dense Neural Network for beat vs non-beat frame classification.

**Datasets:**
- Ballroom Dataset (698 WAV files, 10 genres)
- FMA Small (8000 MP3 files)

**Output:** `beat_detector.h5` — download and place in `backend/app/storage/models/`

## 1. Install Dependencies

In [ ]:
!pip install librosa soundfile tensorflow numpy scipy -q

## 2. Upload Datasets

Upload your datasets to Kaggle as a Dataset, or use the file upload.

Expected paths (adjust if your Kaggle dataset is named differently):
- `/kaggle/input/ballroom-dataset/BallroomData/` — WAV files by genre
- `/kaggle/input/fma-small/fma_small/` — MP3 files in numbered folders

If you uploaded as a zip, uncomment the cells below to extract.

In [ ]:
# # Uncomment if you uploaded zips
# import zipfile
# zipfile.ZipFile('/kaggle/input/your-dataset/BallroomData.zip').extractall('/kaggle/working/BallroomData')
# zipfile.ZipFile('/kaggle/input/your-dataset/fma_small.zip').extractall('/kaggle/working/fma_small')

In [ ]:
import os
from pathlib import Path

# ── Adjust these paths to match your Kaggle dataset ──
BALLROOM_DIR = Path('/kaggle/input/ballroom-dataset/BallroomData')
FMA_DIR = Path('/kaggle/input/fma-small/fma_small')

# Verify
if BALLROOM_DIR.exists():
    wavs = list(BALLROOM_DIR.rglob('*.wav'))
    print(f'Ballroom: {len(wavs)} WAV files')
else:
    print(f'Ballroom not found at {BALLROOM_DIR}')

if FMA_DIR.exists():
    mp3s = list(FMA_DIR.rglob('*.mp3'))
    print(f'FMA Small: {len(mp3s)} MP3 files')
else:
    print(f'FMA not found at {FMA_DIR}')

## 3. Configuration

In [ ]:
# Audio processing parameters
SAMPLE_RATE = 22050
HOP_LENGTH = 512
N_FFT = 2048
N_MFCC = 13

# Training parameters
EPOCHS = 50
BATCH_SIZE = 32
BEAT_THRESHOLD = 0.5

# Limits (set to None for full dataset)
MAX_BALLROOM = None  # e.g. 100 for quick test
MAX_FMA = None

## 4. Feature Extraction Functions

In [ ]:
import numpy as np
import librosa

def load_audio(path, sr=SAMPLE_RATE):
    """Load audio, convert to mono, normalize."""
    y, loaded_sr = librosa.load(path, sr=sr, mono=True)
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    return y, loaded_sr

def extract_features(y, sr):
    """Extract MFCCs, onset strength, spectral flux, energy envelope."""
    # MFCCs
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC,
                                 hop_length=HOP_LENGTH, n_fft=N_FFT).T
    # Onset strength
    onset = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP_LENGTH)
    
    # Spectral flux
    S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
    flux = np.sqrt(np.sum(np.diff(S, axis=1) ** 2, axis=0))
    flux = np.concatenate([[0.0], flux])
    
    # Energy envelope (RMS)
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
    
    # Align to common frame count
    n_frames = min(mfcc.shape[0], len(onset), len(flux), len(rms))
    features = np.column_stack([
        mfcc[:n_frames],
        onset[:n_frames],
        flux[:n_frames],
        rms[:n_frames],
    ])
    return features

def generate_beat_labels(y, sr, n_frames):
    """Generate pseudo beat labels using librosa beat tracker."""
    _, beat_frames = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP_LENGTH)
    labels = np.zeros(n_frames, dtype=np.float32)
    for bf in beat_frames:
        if 0 <= bf < n_frames:
            labels[bf] = 1.0
    return labels

print(f'Feature dimension per frame: {N_MFCC + 3}')

## 5. Process Ballroom Dataset

In [ ]:
all_X = []
all_y = []

if BALLROOM_DIR.exists():
    wav_files = sorted(BALLROOM_DIR.rglob('*.wav'))
    if MAX_BALLROOM:
        wav_files = wav_files[:MAX_BALLROOM]
    
    print(f'Processing {len(wav_files)} Ballroom files...')
    
    for i, wav_path in enumerate(wav_files, 1):
        try:
            y_audio, sr = load_audio(str(wav_path))
            features = extract_features(y_audio, sr)
            labels = generate_beat_labels(y_audio, sr, features.shape[0])
            all_X.append(features)
            all_y.append(labels)
            
            if i % 50 == 0:
                print(f'  Progress: {i}/{len(wav_files)}')
        except Exception as e:
            print(f'  Skipping {wav_path.name}: {e}')
    
    print(f'Ballroom done: {len(all_X)} files processed')
else:
    print('Ballroom dataset not found — skipping')

## 6. Process FMA Small Dataset

In [ ]:
if FMA_DIR.exists():
    mp3_files = sorted(FMA_DIR.rglob('*.mp3'))
    if MAX_FMA:
        mp3_files = mp3_files[:MAX_FMA]
    
    print(f'Processing {len(mp3_files)} FMA files...')
    
    for i, mp3_path in enumerate(mp3_files, 1):
        try:
            y_audio, sr = load_audio(str(mp3_path))
            features = extract_features(y_audio, sr)
            labels = generate_beat_labels(y_audio, sr, features.shape[0])
            all_X.append(features)
            all_y.append(labels)
            
            if i % 100 == 0:
                print(f'  Progress: {i}/{len(mp3_files)}')
        except Exception as e:
            if i <= 5:
                print(f'  Skipping {mp3_path.name}: {e}')
    
    print(f'FMA done. Total files processed: {len(all_X)}')
else:
    print('FMA dataset not found — skipping')

## 7. Combine & Prepare Training Data

In [ ]:
X = np.vstack(all_X)
y = np.concatenate(all_y)

print(f'Combined dataset:')
print(f'  Features (X): {X.shape}')
print(f'  Labels   (y): {y.shape}')
print(f'  Beat ratio:   {y.mean():.4f} ({int(y.sum())} beats / {len(y)} frames)')

# Save processed data
np.save('/kaggle/working/X.npy', X)
np.save('/kaggle/working/y.npy', y)
print('Saved X.npy and y.npy')

## 8. Build & Train ANN Model

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X.shape[1]  # n_mfcc + 3 = 16

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()

In [ ]:
history = model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    verbose=1,
)

print(f'\nFinal training accuracy:   {history.history["accuracy"][-1]:.4f}')
print(f'Final validation accuracy: {history.history["val_accuracy"][-1]:.4f}')

## 9. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()

## 10. Save Model

In [ ]:
model.save('/kaggle/working/beat_detector.h5')
print('Model saved to /kaggle/working/beat_detector.h5')
print('\n=== NEXT STEPS ===')
print('1. Download beat_detector.h5 from the Output tab')
print('2. Place it in: backend/app/storage/models/beat_detector.h5')
print('3. The API will automatically use it for beat detection!')